# T2.1 – Pre-Trained YOLO Baseline on KITTI (Car Class)

## 1. Setup & Imports

In [ ]:
# 1. Install YOLO (You may need to restart your kernel after this runs for the first time)
!pip install -U ultralytics

# Standard & File System Libraries
import os

# Visualization & Display
from IPython.display import Image, display

# Deep Learning / YOLO
from ultralytics import YOLO

from google.colab import drive
drive.mount('/content/drive')

print("Environment setup complete. All libraries imported.")

## 2. Load & Extract the Dataset

In [ ]:
# Replace 'kitti3k_yolo.zip' with the exact name of the zip file if it is different.
# If the zip file is inside a specific folder in Drive, include that folder path
!unzip -q "/content/drive/MyDrive/kitti3k_yolo.zip" -d "/content/kitti3k_yolo/"

print("Unzipping complete!")


### Verify Extracted Files

In [ ]:
# Check if the folder exists and list its contents
if os.path.exists("/content/kitti3k_yolo/"):
    print("Files in the dataset folder:")
    print(os.listdir("/content/kitti3k_yolo/"))
else:
    print("Folder not found. Double-check the zip file name and path.")


Flatten the Dataset Folder Structure

In [ ]:
# Move everything from the inner folder up one level
!mv /content/kitti3k_yolo/kitti3k_yolo/* /content/kitti3k_yolo/

## 3. Evaluate the Pre-Trained Model (Car Class)

In [ ]:
# 1. Load the pre-trained YOLO model
print("Loading pre-trained yolo11n.pt model...")
pretrained_model = YOLO("yolo11n.pt")

# 2. Define the dataset config path
# IMPORTANT: ensure 'dataset_yaml' points to the correct location of the kitti3k_yolo.yaml file
dataset_yaml = "/content/kitti3k_yolo/kitti3k.yaml" # Update this path if the folder structure differs

# 3. Evaluate the pre-trained model on the Car class (classes=[2])
print("Evaluating on the test set for the Car class...")
pretrained_val_metrics = pretrained_model.val(
    data=dataset_yaml,
    split="test",
    classes=[2],
    imgsz=640,
    conf=0.001,
    iou=0.7,
    exist_ok=True
)

# 4. Extract and print the required metrics (AP50 and AP50-95)
# YOLO stores these metrics in the 'box' attribute of the results object
print("\n--- T2.1 Pre-trained Model Results ---")
print(f"Car AP50:    {pretrained_val_metrics.box.map50:.4f}")
print(f"Car AP50-95: {pretrained_val_metrics.box.map:.4f}")
print("--------------------------------------")


## 4. Display Validation Plots

In [ ]:
# Path where YOLO saved the validation artifacts
val_dir = "runs/detect/val"

# Key plots to display (using the current YOLO 'Box' prefixes)
plots_to_show = [
    ("Precision-Recall Curve (Required for T2.1)", "BoxPR_curve.png"),
    ("Confusion Matrix", "confusion_matrix.png"),
    ("F1 Score vs Confidence Curve", "BoxF1_curve.png"),
    ("Sample Predictions", "val_batch0_pred.jpg")
]

for title, file_name in plots_to_show:
    file_path = os.path.join(val_dir, file_name)
    if os.path.exists(file_path):
        print(f"=== {title} ===")
        display(Image(filename=file_path, width=600))
    else:
        print(f"Could not find {file_name} in {val_dir}")
